# Nombre: Gabriel Jaramillo Cuberos (Id: 20529022)

Materia: Procesamiento de Datos (1255)

Tema: Introducción a Apache Spark.

Descripción: En este taller se desarrolla un tutorial para aprender a usar Apache Spark.

Nombre del fichero: ApacheSparkTutorial.ipynb

Taller de Introducción a ApacheSpark/PySpark

# Apache Spark - Beginner Tutorial

Firstly, Thank you for checking this tutorial notebook. By now you must have read a lot about Apache Spark and its ML Library. So, I won't bore you with the introduction to Apache-Spark or even the library details. We shall move straight to the interesting stuff i.e. coding.

In this tuorial notebook we will try to compare some of the Apache Spark's Classification Algorithms in an easy way to make predictions. And since this is a beginner's tutorial, we will use the Iris Flower Dataset aka the beginner's dataset in machine learning.

**A quick summary:**

* Import Libraries
* Build Spark Session
* Data Load
* Data Exploration & Preparation
* Feature Engineering
* Data Scaling
* Data Split
* Build, Train & Evaluate Model


## Importing Libraries

In [11]:
!pip install tabulate

In [12]:
!pip install numpy
!pip install pandas
!pip install pyspark

In [13]:
!pip install scikit-learn

In [4]:
#Generic Libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

#Apache Spark Libraries
import pyspark
from pyspark.sql import SparkSession

#Apache Spark ML CLassifier Libraries
from pyspark.ml.classification import DecisionTreeClassifier,RandomForestClassifier,NaiveBayes

#Apache Spark Evaluation Library
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Apache Spark Features libraries
from pyspark.ml.feature import StandardScaler,StringIndexer, VectorAssembler, VectorIndexer, OneHotEncoder

#Apache Spark Pipelin Library
from pyspark.ml import Pipeline

# Apache Spark `DenseVector`
from pyspark.ml.linalg import DenseVector

#Data Split Libraries
import sklearn
from sklearn.model_selection import train_test_split


#Tabulating Data
#from tabulate import tabulate

#Garbage
import gc

## Build Spark Session

In [ ]:
import sys
import os
import pyspark
from pyspark import SparkConf

print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("PySpark:", pyspark.__version__)
print("Ubicación de PySpark:", pyspark.__file__)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("SPARK_CONF_DIR:", os.environ.get("SPARK_CONF_DIR"))

conf = SparkConf()
for clave in (
    "spark.master",
    "spark.scheduler.mode",
    "spark.scheduler.allocation.file",
):
    print(clave, "=", conf.get(clave, "(sin definir)"))

In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.11"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Apache Spark Beginner Tutorial")
    .master("****")
    .config("spark.pyspark.python", "/usr/bin/python3.11")
    .config(
        "spark.scheduler.allocation.file",
        "file:///opt/cluster/spark/conf/fairscheduler.xml"
    )
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "2")
    .config("spark.cores.max", "2")
    .getOrCreate()
)

print("Python del notebook:", sys.executable)
print("Python para las tareas:", spark.sparkContext.pythonExec)

Python del notebook: /home/estudiante/jupyter-spark-env/bin/python
Python para las tareas: /usr/bin/python3.11


In [7]:
def comprobar_numpy(_):
    import socket
    import sys
    import site
    import importlib.util

    modulo = importlib.util.find_spec("numpy")

    return {
        "equipo": socket.gethostname(),
        "python": sys.executable,
        "paquetes_usuario_habilitados": site.ENABLE_USER_SITE,
        "carpeta_usuario": site.getusersitepackages(),
        "numpy": modulo.origin if modulo else "NO ENCONTRADO",
    }

spark.sparkContext.parallelize([0], 1).map(comprobar_numpy).collect()

[{'equipo': 'NBDG49',
  'python': '/usr/bin/python3.11',
  'paquetes_usuario_habilitados': True,
  'carpeta_usuario': '/home/estudiante/.local/lib/python3.11/site-packages',
  'numpy': '/home/estudiante/.local/lib/python3.11/site-packages/numpy/__init__.py'}]

In [8]:
def probar_vector(_):
    import numpy
    from pyspark.ml.linalg import DenseVector

    return numpy.__version__, str(DenseVector([1.0, 2.0]))

spark.sparkContext.parallelize([0], 1).map(probar_vector).collect()

[('2.0.2', '[1.0,2.0]')]

In [9]:
def revisar_python(_):
    import sys
    import socket
    return socket.gethostname(), sys.executable, sys.version.split()[0]

spark.sparkContext.parallelize([0], 1).map(revisar_python).collect()

[('NBDG49', '/usr/bin/python3.11', '3.11.13')]

In [4]:
#Celda contiene malas configuraciones, celda de arriba funciona correctamente.

#Building Spark Session
#spark = (SparkSession.builder
#                  .appName('Apache Spark Beginner Tutorial')
#                  .config("spark.executor.memory", "1G")
#                  .config("spark.executor.cores","4")
#                  .getOrCreate())

In [10]:
spark.sparkContext.setLogLevel('INFO')
current_level = spark._jvm.org.apache.log4j.Logger.getRootLogger().getLevel()

#Comprobación
print(f"Nivel: {current_level}")

Nivel: INFO


In [11]:
spark.version

'4.2.0'

## Data Load

In [ ]:
# Codigo antiguo:

#url = 'iris.csv'
#
#data = spark.read.format("csv").option("header", "true").option("inferSchema","true").load(url) 
#
#data.cache() #for faster re-use

# Codigo Nuevo:

data = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("file:///opt/cluster/documentos/iris.csv") # Establecer ruta del archivo
)

data.cache()
data.show(5)

# Explicacion del error: Spark asigna la tarea de buscar el archivo a cualquier worker del cluster que se encuentre activo. Por lo tanto, todos los workers deben tener el archivo en la misma ubicación para evitar errores de lectura.

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows


## Data Exploration & Preparation

In [13]:
#Total records 
data.count()

150

In [14]:
#Data Type
data.printSchema()

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)



In [15]:
#Display records
data.show(5)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows


In [16]:
#Records per Species
data.groupBy('species').count().show()

# Bloque hace recuento de la cantidad de individuos registrada en cada especie.

+----------+-----+
|   species|count|
+----------+-----+
| virginica|   50|
|versicolor|   50|
|    setosa|   50|
+----------+-----+



In [17]:
#Dataset Summary Stats
data.describe().show()

# La salida muestra una advertencia de la longitud de la cadena de representación. Sin embargo, Jupyter ajusta la cadena, lo que no supone problemas para la ejecución.

26/09/17 14:29:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 15:>                                                         (0 + 1) / 1]

+-------+------------------+-------------------+------------------+------------------+---------+
|summary|      sepal_length|        sepal_width|      petal_length|       petal_width|  species|
+-------+------------------+-------------------+------------------+------------------+---------+
|  count|               150|                150|               150|               150|      150|
|   mean| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|     NULL|
| stddev|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|     NULL|
|    min|               4.3|                2.0|               1.0|               0.1|   setosa|
|    max|               7.9|                4.4|               6.9|               2.5|virginica|
+-------+------------------+-------------------+------------------+------------------+---------+



Inorder for our model to make predictions the Species aka Label column should be a numerical value (models don't like string!). To achieve this we shall use String Indexing on the Species columns

In [18]:
#String Indexing the Species column
SIndexer = StringIndexer(inputCol='species', outputCol='species_indx')
data = SIndexer.fit(data).transform(data)

#Inspect the dataset
data.show(5)

# Este bloque obtiene los valores de la columna "Especie" y genera un índice numérico para poder representar la especie.

+------------+-----------+------------+-----------+-------+------------+
|sepal_length|sepal_width|petal_length|petal_width|species|species_indx|
+------------+-----------+------------+-----------+-------+------------+
|         5.1|        3.5|         1.4|        0.2| setosa|         0.0|
|         4.9|        3.0|         1.4|        0.2| setosa|         0.0|
|         4.7|        3.2|         1.3|        0.2| setosa|         0.0|
|         4.6|        3.1|         1.5|        0.2| setosa|         0.0|
|         5.0|        3.6|         1.4|        0.2| setosa|         0.0|
+------------+-----------+------------+-----------+-------+------------+
only showing top 5 rows


## Feature Engineering

The Spark model needs two columns: “label” and “features” and we are not going to do much feature engineering because we want to focus on the mechanics of training the model in Spark. 

So, creating a seperate dataframe with re-ordered columns, then defining an input data using Dense Vector. A Dense Vector is a local vector that is backed by a double array that represents its entry values. In other words, it's used to store arrays of values for use in PySpark.


In [19]:
#creating a seperate dataframe with re-ordered columns
df = data.select("species_indx","sepal_length", "sepal_width", "petal_length", "petal_width")

#Inspect the dataframe
df.show(5)

# Este bloque genera una nueva fuente de datos a partir de la fuente anterior, usando exclusivamente valores numéricos y quitando cualquier cadena de texto.

+------------+------------+-----------+------------+-----------+
|species_indx|sepal_length|sepal_width|petal_length|petal_width|
+------------+------------+-----------+------------+-----------+
|         0.0|         5.1|        3.5|         1.4|        0.2|
|         0.0|         4.9|        3.0|         1.4|        0.2|
|         0.0|         4.7|        3.2|         1.3|        0.2|
|         0.0|         4.6|        3.1|         1.5|        0.2|
|         0.0|         5.0|        3.6|         1.4|        0.2|
+------------+------------+-----------+------------+-----------+
only showing top 5 rows


**Note:** Observe that the species column which is our label (aka Target) is now at beginning of the dataframe

In [22]:
# Define the `input_data` as Dense Vector
input_data = df.rdd.map(lambda x: (x[0], DenseVector(x[1:])))

# Este bloque obtiene los registros y los guarda como tuplas. Cada tupla esta compuesta por un identificador, que es el x[0], y un vector denso que contiene toda la info. siguiente al identificador. 
# Esto se hace porque los modelos de ML necesitan variables que estén dentro de un solo vector de características. No aceptan columnas sueltas.

**Note:** Observe the definition of the Dense Vector. So,when we create a new indexed dataframe(below) the machine understands that the first column is a Label (Target) and the remaining columns are Features.

In [23]:
import sys
import socket
import pyspark
print("Equipo del notebook:", socket.gethostname())
print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("PySpark:", pyspark.__version__)

Equipo del notebook: NBDG62
Python: 3.11.13 (main, Jun  1 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]
Ejecutable: /home/estudiante/jupyter-spark-env/bin/python
PySpark: 4.2.0


In [24]:
# Creating a new Indexed Dataframe
df_indx = spark.createDataFrame(input_data, ["label", "features"])

# Este bloque genera un nuevo dataframe con los vectores generados en el bloque pasado.

In [25]:
#view the indexed dataframe
df_indx.show(5)

[Stage 27:>                                                         (0 + 1) / 1]

+-----+-----------------+
|label|         features|
+-----+-----------------+
|  0.0|[5.1,3.5,1.4,0.2]|
|  0.0|[4.9,3.0,1.4,0.2]|
|  0.0|[4.7,3.2,1.3,0.2]|
|  0.0|[4.6,3.1,1.5,0.2]|
|  0.0|[5.0,3.6,1.4,0.2]|
+-----+-----------------+
only showing top 5 rows


## Error:
El nuevo dataframe con los vectores no podía ser creado porque el worker 49 usaba una versión antigua de PySpark. Para la solución, fue necesario crear y registrar un nuevo kernel con la versión de PySpark actualizada.

## Data Scaling

This is also known as Feature Scaling. It is a method of normalizing the features of the data. Scaling can make a difference between a weak machine learning model and a better one. 

In this tutorial we will use a Standard Scaler to scale our feature data. Apache Spark has a Standard Scaler library to do the job.

In [26]:
#Initialize Standard Scaler
stdScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

#Fit the Standard Scaler to the indexed Dataframe
scaler = stdScaler.fit(df_indx)

#Transform the dataframe
df_scaled =scaler.transform(df_indx)

# Este bloque se encarga de escalar los datos del dataframe creado anteriormente con el objetivo de tener un mejor módelo de entrenamiento.

In [27]:
#Viewing the Scaled Data
df_scaled.show(5)

+-----+-----------------+--------------------+
|label|         features|     features_scaled|
+-----+-----------------+--------------------+
|  0.0|[5.1,3.5,1.4,0.2]|[6.15892840883878...|
|  0.0|[4.9,3.0,1.4,0.2]|[5.9174018045706,...|
|  0.0|[4.7,3.2,1.3,0.2]|[5.67587520030241...|
|  0.0|[4.6,3.1,1.5,0.2]|[5.55511189816831...|
|  0.0|[5.0,3.6,1.4,0.2]|[6.03816510670469...|
+-----+-----------------+--------------------+
only showing top 5 rows


In [29]:
#Dropping the Features column
df_scaled = df_scaled.drop("features")

# Mostrar sin la columna de características, eliminada previamente.
df_scaled.show(5)

[Stage 32:>                                                         (0 + 1) / 1]

+-----+--------------------+
|label|     features_scaled|
+-----+--------------------+
|  0.0|[6.15892840883878...|
|  0.0|[5.9174018045706,...|
|  0.0|[5.67587520030241...|
|  0.0|[5.55511189816831...|
|  0.0|[6.03816510670469...|
+-----+--------------------+
only showing top 5 rows


## Data Split

Just like always, before building a model we shall split our scaled dataset into training & test sets. 
Training Dataset = 90%
Test Dataset = 10%

In [30]:
train_data, test_data = df_scaled.randomSplit([0.9, 0.1], seed = 12345)

# Este bloque se encarga de dividir los datos aleatoriamente, de los cuales el 90% van a ser usados para entrenamiento y el 10% restante para pruebas.

In [31]:
#Inspect Training Data
# Muestra los datos de entrenamiento
train_data.show(5)

[Stage 33:>                                                         (0 + 1) / 1]

+-----+--------------------+
|label|     features_scaled|
+-----+--------------------+
|  0.0|[5.19282199176603...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.43434859603422...|
+-----+--------------------+
only showing top 5 rows


## Build, Train & Evaluate Model

In this step we will create multiple models, train them on our scaled dataset and then compare their accuracy.

In [32]:
model = ['Decision Tree','Random Forest','Naive Bayes']
model_results = []

# Crea los contenedores para los modelos.

In [39]:
# -- Decision Tree Classifier --

dtc = DecisionTreeClassifier(labelCol="label", featuresCol="features_scaled")          #instantiate the model
dtc_model = dtc.fit(train_data)                                                        #train the model
dtc_pred = dtc_model.transform(test_data)                                              #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
dtc_acc = evaluator.evaluate(dtc_pred)
print("Decision Tree Classifier Accuracy =", '{:.2%}'.format(dtc_acc))
model_results.extend([[model[0],'{:.2%}'.format(dtc_acc)]])                               #appending to list

# Genera un modelo clasificador como un árbol de decisiones. Guarda resultados en el contenedor creado previamente.

# Mostrar progreso
model_results

Decision Tree Classifier Accuracy = 90.91%


[['Decision Tree', '90.91%'],
 ['Decision Tree', '90.91%'],
 ['Random Forest', '100.00%'],
 ['Naive Bayes', '100.00%'],
 ['Decision Tree', '90.91%']]

In [40]:
# -- Random Forest Classifier --

rfc = RandomForestClassifier(labelCol="label", featuresCol="features_scaled", numTrees=10)          #instantiate the model
rfc_model = rfc.fit(train_data)                                                                     #train the model
rfc_pred = rfc_model.transform(test_data)                                                           #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
rfc_acc = evaluator.evaluate(rfc_pred)
print("Random Forest Classifier Accuracy =", '{:.2%}'.format(rfc_acc))
model_results.extend([[model[1],'{:.2%}'.format(rfc_acc)]])                                            #appending to list

# Genera un modelo bajo el algoritmo Random Forest. Guarda resultados en el contenedor creado previamente.

# Mostrar progreso
model_results

Random Forest Classifier Accuracy = 100.00%


[['Decision Tree', '90.91%'],
 ['Decision Tree', '90.91%'],
 ['Random Forest', '100.00%'],
 ['Naive Bayes', '100.00%'],
 ['Decision Tree', '90.91%'],
 ['Random Forest', '100.00%']]

In [41]:
# -- Naive Bayes Classifier --

nbc = NaiveBayes(smoothing=1.0,modelType="multinomial", labelCol="label",featuresCol="features_scaled")    #instantiate the model
nbc_model = nbc.fit(train_data)                                                                          #train the model
nbc_pred = nbc_model.transform(test_data)                                                                #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
nbc_acc = evaluator.evaluate(nbc_pred)
print("Naive Bayes Accuracy =", '{:.2%}'.format(nbc_acc))
model_results.extend([[model[2],'{:.2%}'.format(nbc_acc)]])                                            #appending to list

# Genera un clasificador bajo el método bayesiano ingenuo. Guarda resultados en el contenedor creado previamente.

# Mostrar progreso
model_results

Naive Bayes Accuracy = 100.00%


[['Decision Tree', '90.91%'],
 ['Decision Tree', '90.91%'],
 ['Random Forest', '100.00%'],
 ['Naive Bayes', '100.00%'],
 ['Decision Tree', '90.91%'],
 ['Random Forest', '100.00%'],
 ['Naive Bayes', '100.00%']]

In [42]:
#freeing memory
gc.collect()

# Libera memoria.

451

Tabulating the results.

# Final y conclusiones

- Tanto el modelo bayesiano, como el modelo Random Forest, evidencian ser mas fuertes que el árbol de decisión ya que se puede apreciar una exactitud del 100%, mientras que el tercero muestra una del 90%.
- La instalación de Jupyter en una máquina virtual permitió ejecutar cuadernos Python desde el navegador y acceder de forma remota mediante un túnel SSH.
- La configuración de Spark mostró la importancia de conectar correctamente el master y los workers, verificar sus direcciones y distribuir los recursos disponibles.
- Mantener versiones compatibles de Python, PySpark y las librerías en los equipos es fundamental para evitar errores durante la ejecución.
- En un clúster, los archivos deben estar disponibles para todos los workers, mediante copias locales o almacenamiento compartido como HDFS.
- Spark permite distribuir el procesamiento entre varios equipos, lo que facilita trabajar con grandes volúmenes de datos. Sin embargo, su buen funcionamiento depende de una configuración adecuada del entorno y los recursos.

![](https://www.appreciationatwork.com/wp-content/uploads/2018/01/thank-you.jpg)

I hope this tutorial was helpful.